In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedModelInput, TrafficConfig, Route

def tool_configure_ab_test_traffic(endpoint_name: str, champion_model_version: str, challenger_model_version: str, challenger_traffic_pct: int = 10):
    """Configura o roteamento de tráfego para Teste A/B no Databricks Model Serving."""
    w = WorkspaceClient()
    
    champion_traffic = 100 - challenger_traffic_pct
    
    served_models = [
        ServedModelInput(
            name="champion",
            model_name=champion_model_version.split("@")[0],
            model_version=champion_model_version.split("@")[1] if "@" in champion_model_version else "1",
            workload_size="Small",
            scale_to_zero_enabled=True,
        ),
        ServedModelInput(
            name="challenger",
            model_name=challenger_model_version.split("@")[0],
            model_version=challenger_model_version.split("@")[1] if "@" in challenger_model_version else "2",
            workload_size="Small",
            scale_to_zero_enabled=True,
        )
    ]

    traffic_config = TrafficConfig(routes=[
        Route(served_model_name="champion", traffic_percentage=champion_traffic),
        Route(served_model_name="challenger", traffic_percentage=challenger_traffic_pct),
    ])
    
    print(f"[Agente] Configurando Teste A/B no Endpoint '{endpoint_name}':")
    print(f" - Champion: {champion_traffic}%")
    print(f" - Challenger: {challenger_traffic_pct}%")
    
    # Exemplo de chamada à API do Databricks Serving:
    # w.serving_endpoints.update(name=endpoint_name, config=EndpointCoreConfigInput(name=endpoint_name, served_models=served_models, traffic_config=traffic_config))
    
    return {"status": "ab_test_configured", "endpoint": endpoint_name, "split": f"{champion_traffic}/{challenger_traffic_pct}"}